In [1]:
%run common_setup.ipynb

#### This cell reads the RAW table of all WORKS and computes work-level properties. 

-  UNNEST authorships to author and institution  
-  UNNEST reference_list to citer_id -> cited_id  
-  COMPUTE work-level properties 
    - page count and references per page (many page counts are unknown)  
    - citation count from the endogenous set  
    - self-cited works  
    - number of copied references (one level deep)


In [2]:
class TransformWorks(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def transform_works(self):

        sql = """
            CREATE OR REPLACE TABLE project.works_transform AS  
            WITH
            unnest_authorships_CTE AS
                (SELECT id AS work_id,
                        unnest(authorships) AS authorship,
                    FROM project.raw
                -- LIMIT 128
                ),
            unnest_institutions_CTE AS
                (SELECT work_id,
                        authorship,
                        unnest(authorship.institutions) AS institution
                    FROM unnest_authorships_CTE
                ),
            assemble_author_institution_CTE AS
                (SELECT work_id,
                        authorship.author.id AS author_id,
                        institution.id AS institution_id,
                        institution.country_code AS country_code
                    FROM unnest_institutions_CTE
                ),
            assemble_work_author_institution_CTE AS
                (SELECT r.id AS work_id,
                        r.publication_year,
                        r.cited_by_count AS cited_by_count_oa,
                        try_cast("biblio.last_page" AS INT) - try_cast("biblio.first_page" AS INT)+1 AS page_length,
                        round(referenced_works_count/page_length, 2) AS references_per_page,
                        author_id,
                        institution_id,
                        country_code
                    FROM project.raw r
                    INNER JOIN assemble_author_institution_CTE aai
                    ON r.id = aai.work_id
                ),
            extract_references_CTE AS
                (SELECT id AS citer_id,
                        unnest(referenced_works) AS cited_id,
                    FROM project.raw
                ),
            get_citation_count_endogenous_CTE AS
                (SELECT cited_id,
                        count(citer_id) AS cited_by_count_endogenous
                    FROM extract_references_CTE
                    GROUP BY ALL
                    ORDER BY cited_by_count_endogenous DESC
                ),
            get_selfcited_CTE AS
                (SELECT cited_id,
                        isSelfCited
                    FROM
                    (SELECT citer_id,
                            cited_id,
                            CASE WHEN list_has_any(citer_authors, cited_authors) = true IS true THEN 1 ELSE NULL END AS isSelfCited
                    FROM 
                        (SELECT citer_id, 
                                list(citer.author_id) AS citer_authors,
                                cited_id,
                                list(cited.author_id) AS cited_authors
                            FROM extract_references_CTE
                            INNER JOIN assemble_work_author_institution_CTE citer
                            ON citer_id = citer.work_id
                            INNER JOIN assemble_work_author_institution_CTE cited
                            ON cited_id = cited.work_id
                            GROUP BY citer_id, cited_id
                        )
                    )
                ),
            get_copied_CTE AS
                (SELECT r1.citer_id,
                        referenced_works_count,
                        count(r2.cited_id) AS copied_count,
                    FROM extract_references_CTE r1
                    LEFT JOIN extract_references_CTE r2
                    ON r1.cited_id = r2.citer_id
                    LEFT JOIN project.raw
                    ON id = r1.citer_id
                    WHERE r1.cited_id = r2.cited_id
                    GROUP BY ALL
                ),
            get_single_works_CTE AS
                (SELECT r.id AS work_id,
                        r.publication_year,
                        r."primary_location.source".id AS source_id,
                        r."primary_location.source".host_organization AS host_id,
                        r.cited_by_count AS cited_by_count_oa,
                        e.cited_by_count_endogenous,
                        try_cast("biblio.last_page" AS INT) - try_cast("biblio.first_page" AS INT)+1 AS page_length,
                        round(c.referenced_works_count/page_length, 2) AS references_per_page,
                        c.referenced_works_count,
                        c.copied_count,
                        c.copied_count/c.referenced_works_count AS copied_fraction,
                        s.isSelfCited,
                        count(a.authorship.author.id) AS authors_distinct_count,
                        institutions_distinct_count,
                        countries_distinct_count
                    FROM project.raw r
                    LEFT JOIN get_copied_CTE c
                    ON r.id = c.citer_id
                    LEFT JOIN get_selfcited_CTE s
                    ON r.id = s.cited_id
                    LEFT JOIN get_citation_count_endogenous_CTE e
                    ON r.id = e.cited_id
                    LEFT JOIN unnest_authorships_CTE a
                    ON r.id = a.work_id
                    GROUP BY ALL
                )

            SELECT *
            FROM get_single_works_CTE
            ORDER BY copied_fraction DESC
        """
        self.db.sql(sql)
        df = self.db.sql("SELECT * FROM project.works_transform").df()
        print(f'{df.shape = }\n{df.head()}')
        return

#### This cell explores filters for less-significant institutions and authors. 

- Their removal:
  - Simplifies interpretations
  - Speeds calculations
  - For PageRank, more closely symmetrises node counts for journals and institutions when these are combined 


In [3]:
class FilterNew(SetUp):

    def __init__(self):
        super().__init__()

        sql = """  

            -- EXAMINE FILTERING OF INSTITUTIONS, AUTHORS, JOURNALS AND WORKS
            -- ==============================================================
            WITH 
            unnest_authorship_CTE AS
                (SELECT id,
                        unnest(authorship.institutions).id AS institution_id
                ),
            unnest_authorships_CTE AS
                (SELECT id,
                        "primary_location.source".id AS source_id,
                        authorship.author.id AS author_id,
                        authorship, 
                        publication_year
                FROM project.raw 
                LEFT JOIN (SELECT id, unnest(authorships) AS authorship FROM project.raw)
                USING (id)
                ),
            raw_expanded_CTE AS
                (SELECT DISTINCT id,
                        source_id,
                        author_id,
                        institution_id,
                        publication_year,
                        -- count(id) OVER (), 
                        -- count(DISTINCT id) OVER (),
                        -- count(DISTINCT source_id) OVER (),
                        -- count(DISTINCT author_id) OVER (),
                        -- count(DISTINCT institution_id) OVER ()
                FROM unnest_authorships_CTE
                LEFT JOIN unnest_authorship_CTE
                USING (id)
                WHERE author_id IS NOT NULL AND institution_id IS NOT NULL
                ),
            partitioned_counts_CTE AS
                (SELECT source_id,
                        count(DISTINCT id) OVER (PARTITION BY source_id) AS source_counts,
                        author_id,
                        count(DISTINCT ID) OVER (PARTITION BY author_id) AS author_counts,
                        institution_id,
                        count(DISTINCT id) OVER (PARTITION BY institution_id) AS institution_counts,
                FROM raw_expanded_CTE
                ORDER BY source_counts DESC, author_counts DESC, institution_counts DESC
                ),
            filtered_institutions_CTE AS
                (SELECT DISTINCT institution_id,
                        institution_counts
                FROM
                    (SELECT count(DISTINCT id) OVER (PARTITION BY institution_id) AS institution_counts,
                    institution_id,
                    FROM raw_expanded_CTE
                    )
                WHERE institution_counts >= 10*15
                ORDER BY institution_counts DESC
                ),
            filtered_authors_CTE AS
                (SELECT DISTINCT author_id,
                    author_counts,
                    list_reverse_sort(year_list)[1] -
                    list_sort(year_list)[1] + 1 AS delta_time,
            
                FROM
                    (SELECT count(DISTINCT id) OVER (PARTITION BY author_id) AS author_counts,
                            list(publication_year) OVER (PARTITION BY author_id) AS year_list,
                        author_id,
                    FROM raw_expanded_CTE
                    )
                WHERE author_counts >= 2*delta_time
                ORDER BY delta_time ASC
                )

            -- SELECT list(author_id) FROM filtered_authors_CTE
            SELECT id,
                    source_id,
                    author_id,
                    institution_id,
                    -- count(id) OVER (), 
                    -- count(DISTINCT id) OVER (),
                    -- count(DISTINCT source_id) OVER (),
                    -- count(DISTINCT author_id) OVER (),
                    -- count(DISTINCT institution_id) OVER ()
            FROM raw_expanded_CTE
                LEFT JOIN 
                (SELECT id,
                        list(author_id) AS author_list,
                        list(institution_id) AS institution_list
                    FROM raw_expanded_CTE
                    GROUP BY id
                )
            USING (id)
            WHERE list_has_all((SELECT list(author_id) FROM filtered_authors_CTE), author_list ) AND
                    list_has_all((SELECT list(institution_id) FROM filtered_institutions_CTE), institution_list )
            ORDER BY id
            """
        return

In [4]:
class Filter(SetUp): # type: ignore

    def __init__(self):
        super().__init__()
        return
    
    def filter(self, cutoff_institutions=None, cutoff_authors=None):
        self._filter_institutions(cutoff=cutoff_institutions)
        self._filter_authors(cutoff=cutoff_authors)
        self._modify_authorships()
        self._modify_works()
        self._modify_cites()
        return self
    
    def _filter_institutions(self, cutoff=None):
        sql = f"""
            CREATE OR REPLACE TABLE memory.institutions AS        
                WITH
                get_work_institution_CTE AS
                    (SELECT id AS work_id,
                            unnest(authorships) 
                        FROM project.raw
                    ),
                # get_work_institution_CTE AS (
                #     SELECT DISTINCT work_id, institution_id 
                #     FROM project.authorships_full
                # ),
                institution_counts AS (
                    SELECT
                        institution_id,
                        COUNT(work_id) AS works_count
                    FROM get_work_institution_CTE
                    GROUP BY institution_id
                ),
                ranked_institutions AS (
                    SELECT
                        ROW_NUMBER() OVER (ORDER BY works_count DESC) AS row_number,
                        institution_id,
                        works_count
                    FROM institution_counts
                )

                SELECT *
                    FROM ranked_institutions
                    WHERE works_count >= {cutoff}
                    ORDER BY works_count DESC
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_institutions FROM memory.institutions").show()
        self.db.sql("SELECT count(DISTINCT institution_id) AS count_institutions_original FROM project.authorships_full").show()
        return self

    def _filter_authors(self, cutoff=None):
        sql = f"""
            CREATE OR REPLACE TABLE memory.authors AS
                WITH get_work_author_CTE AS (
                    SELECT DISTINCT work_id, author_id, list_sort(list(publication_year))[1] AS first_year
                    FROM project.authorships_full
                    -- LEFT JOIN project.works_full
                    LEFT JOIN project.works_transform
                    USING (work_id)
                    GROUP BY work_id, author_id
                ),
                author_counts_CTE AS (
                    SELECT
                        author_id,
                        COUNT(work_id)/(2026-first_year) AS works_count_prorata
                    FROM get_work_author_CTE
                    GROUP BY author_id, first_year
                ),
                ranked_authors_CTE AS (
                    SELECT
                        ROW_NUMBER() OVER (ORDER BY works_count_prorata DESC) AS row_number,
                        author_id,
                        works_count_prorata
                    FROM author_counts_CTE
                )

                SELECT *
                FROM ranked_authors_CTE
                WHERE works_count_prorata >= {cutoff}
                ORDER BY works_count_prorata DESCSELECT * 
        --   FROM sources_CTE
                """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_authors FROM memory.authors").show()
        self.db.sql("SELECT count(DISTINCT author_id) AS count_authors_original FROM project.authorships_full").show()
        return self
    
    def _modify_authorships(self):
        sql = """
            CREATE OR REPLACE TABLE memory.authorships ASSELECT * 
            FROM sources_CTE
            WITH
            filtered_works_authors_CTE AS
                (SELECT a.* 
                    FROM project.authorships_full a
                    WHERE author_id IN (SELECT author_id FROM memory.authors))

            SELECT a.* 
                FROM filtered_works_authors_CTE a
                WHERE institution_id IN (SELECT institution_id FROM memory.institutions)
            """   

    def _modify_works(self):
        sql = """
            CREATE OR REPLACE TABLE memory.works AS
            SELECT DISTINCT w.* 
                FROM project.works_full w
                WHERE work_id in (SELECT work_id FROM memory.authorships)
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_works FROM memory.works").show()
        self.db.sql("SELECT count(DISTINCT work_id) AS count_works_original FROM project.works").show()
        return self

    def _modify_cites(self):
        sql = """
            CREATE OR REPLACE TABLE memory.cited AS
            WITH
            citer_cited_CTE AS
                (SELECT work_id, 
                        unnest(referenced_works) AS cited_id
                    FROM project.cited_full
                ),
            filter_citer_cited_CTE AS
                (SELECT work_id,
                        cited_id
                    FROM citer_cited_CTE
                    WHERE cited_id IN (SELECT work_id FROM memory.works))

            SELECT work_id, 
                    list(cited_id)
                FROM filter_citer_cited_CTE
                WHERE work_id in (SELECT work_id FROM memory.works)
                GROUP BY work_id
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_citers FROM memory.cited").show()
        self.db.sql("SELECT count(DISTINCT work_id) AS count_citers_original FROM project.cited").show() 
        return

In [5]:
def main():

    tw = TransformWorks()
    tw.transform_works()
    tw.db.close()

    # f = Filter()
    # f.filter(cutoff_institutions=150, cutoff_authors=2)
    # f.db.close()


    return

In [6]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ backup   │ main    │ authors_full         │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR]  │ false     │
│ backup   │ main    │ citer_cited          │ [citer_id, citer_y…  │ [VARCHAR, BIGINT, VARCHAR, BIGINT, …  │ false     │
│ backup   │ main    │ raw                  │ [id, doi, title, p…  │ [VARCHAR, VARCHAR, VARCHAR, BIGINT,…  │ false     │
│ backup   │ main    │ sources_o